# Lessons for Gemma 4 Developer Agent competitors

**Author:** Moran Zavdi · **License:** Apache-2.0

A short, evidence-backed field guide for other competitors. Mount the companion dataset below.

**GitHub (source of truth):** https://github.com/Aeterna-Data-AI-ML-Research/Kaggle-Gemma-4-Developer-Agent-Competition

## Credits & related assets

**Author:** Moran Zavdi · **License:** Apache-2.0

### Canonical source (GitHub)
- **Repository (archived evidence release):** https://github.com/Aeterna-Data-AI-ML-Research/Kaggle-Gemma-4-Developer-Agent-Competition
- **Article (`PAPER.md`):** https://github.com/Aeterna-Data-AI-ML-Research/Kaggle-Gemma-4-Developer-Agent-Competition/blob/main/PAPER.md
- **Pages one-pager:** https://aeterna-data-ai-ml-research.github.io/Kaggle-Gemma-4-Developer-Agent-Competition/
- **Medium:** https://medium.com/@when1go2market

### This study on Kaggle
- **Dataset (required input):** [gemma4-agent-behavior-vs-accuracy](https://www.kaggle.com/datasets/moranzavdi/gemma4-agent-behavior-vs-accuracy) — `moranzavdi/gemma4-agent-behavior-vs-accuracy`
- **Reproduce notebook:** [reproduce-gemma4-agent-behavior-study](https://www.kaggle.com/code/moranzavdi/reproduce-gemma4-agent-behavior-study)
- **Explore notebook:** [explore-gemma4-agent-calibration](https://www.kaggle.com/code/moranzavdi/explore-gemma4-agent-calibration)
- **Lessons notebook:** [gemma4-agent-lessons-for-competitors](https://www.kaggle.com/code/moranzavdi/gemma4-agent-lessons-for-competitors) ← *you are here*

### Related paper-track assets (also by Moran Zavdi)
- **Kaggle paper writeup:** [From Local Validation to Live Evaluation](https://www.kaggle.com/competitions/gemma-4-developer-agent-paper/writeups/from-local-validation-to-live-evaluation)
- **Dataset:** [gemma4-soundness-and-punica-bench](https://www.kaggle.com/datasets/moranzavdi/gemma4-soundness-and-punica-bench)
- **Notebook:** [from-local-validation-to-live-evaluation](https://www.kaggle.com/code/moranzavdi/from-local-validation-to-live-evaluation)

### Competition
- [Google — The Gemma 4 Developer Agent Competition](https://www.kaggle.com/competitions/gemma-4-developer-agent)

Please cite the GitHub release / `CITATION.cff` if you reuse numbers or code.


In [ ]:
from pathlib import Path
import json, tarfile
import pandas as pd

CANDIDATES = [
    Path('/kaggle/input/gemma4-agent-behavior-vs-accuracy'),
    Path('/kaggle/input/datasets/moranzavdi/gemma4-agent-behavior-vs-accuracy'),
    Path('.').resolve(),
    Path('..').resolve(),
]

def has_evidence(p: Path) -> bool:
    return (p / 'evidence' / 'phase1_calibration_900.jsonl').exists()

ROOT = next((p for p in CANDIDATES if has_evidence(p)), None)
if ROOT is None:
    for base in CANDIDATES:
        if not base.exists():
            continue
        work = Path('/kaggle/working/gemma4_unpack')
        work.mkdir(parents=True, exist_ok=True)
        for tar_path in base.glob('*.tar'):
            with tarfile.open(tar_path) as tf:
                tf.extractall(work)
        if has_evidence(work):
            ROOT = work
            break
assert ROOT is not None
print('ROOT =', ROOT)


## Lesson 1 — Public score is a lattice, not a continuous meter

One extra public solve ≈ **0.017** on a two-decimal leaderboard with ~58 tasks.


In [ ]:
lattice_path = ROOT / 'reports' / 'lb_lattice.csv'
if not lattice_path.exists():
    lattice_path = ROOT / 'lb_lattice.csv'
df = pd.read_csv(lattice_path)
display(df)
df['delta_vs_prev'] = df['public_score'].diff()
print('median score step ≈', df['delta_vs_prev'].dropna().median())


## Lesson 2 — Think-1024 changed behavior, not WT accuracy

Pinned result from the study: **48/150 = 48/150**, flips +9/−9.


In [ ]:
def load_jsonl(p):
    return [json.loads(x) for x in Path(p).read_text().splitlines() if x.strip()]

p1 = load_jsonl(ROOT / 'evidence' / 'phase1_calibration_900.jsonl')
p3 = load_jsonl(ROOT / 'evidence' / 'phase3_thinkcap_150.jsonl')
base = [e for e in p1 if e['arm']=='tr20' and e['seed']==1]
cap = p3
assert len(base)==150 and len(cap)==150

def row(label, rows):
    return dict(
        label=label,
        wt=sum(e['pass'] for e in rows),
        strict=sum(e['strict_pass'] for e in rows),
        submitted=sum(e['end_reason']=='submitted' for e in rows),
        thoughts_gt_1024=sum(e['thoughts_gt_1024'] for e in rows),
        time_budget=sum(e['end_reason']=='time_budget' for e in rows),
    )
summary = pd.DataFrame([row('tr20_s1_budget4096', base), row('think1024', cap)])
display(summary)
print('WT delta tasks:', summary.loc[1,'wt'] - summary.loc[0,'wt'])


## Lesson 3 — Empty-diff timeouts usually already found the gold file

Do not spend the campaign optimizing “better search” alone if your failures look like this.


In [ ]:
fatal = [e for e in base if e['end_reason']=='time_budget' and not e['pass'] and e.get('patch_chars',0)==0]
print('empty-diff timeouts on tr20 seed1:', len(fatal))
print('opened gold via read:', sum(e['opened_gold_file_read'] for e in fatal), '/', len(fatal))
print('opened all gold files:', sum(e['opened_all_gold_files'] for e in fatal), '/', len(fatal))


## Lesson 4 — Offline WT can be flat while LB moves

Phase-1 D69 / tr20 / scale10 were statistically tied offline (~30–32% WT) despite
public scores 0.18 / 0.17 / 0.13. Reproduce the full pin gate:


In [ ]:
import subprocess, sys
script = ROOT / 'scripts' / 'reproduce_all_tables.py'
assert script.exists(), script
r = subprocess.run([sys.executable, str(script)], cwd=str(ROOT), capture_output=True, text=True)
print(r.stdout[-2000:] if r.stdout else '')
if r.returncode != 0:
    print(r.stderr)
assert r.returncode == 0
print('ALL_PINNED_CLAIMS_PASS')


## What *not* to expect from this release

- No adapter weights / submission zips
- No sealed-test graded trajectories
- No proprietary prompt grafts

Use this package to **measure** before you spend more GPU on behavior tweaks that
do not move WT.


## Next steps & credits

| Go next | Link |
| --- | --- |
| GitHub repo | https://github.com/Aeterna-Data-AI-ML-Research/Kaggle-Gemma-4-Developer-Agent-Competition |
| Dataset | https://www.kaggle.com/datasets/moranzavdi/gemma4-agent-behavior-vs-accuracy |
| Reproduce | https://www.kaggle.com/code/moranzavdi/reproduce-gemma4-agent-behavior-study |
| Explore | https://www.kaggle.com/code/moranzavdi/explore-gemma4-agent-calibration |
| Lessons | https://www.kaggle.com/code/moranzavdi/gemma4-agent-lessons-for-competitors |
| **Kaggle paper writeup** | https://www.kaggle.com/competitions/gemma-4-developer-agent-paper/writeups/from-local-validation-to-live-evaluation |
| Paper-track dataset | https://www.kaggle.com/datasets/moranzavdi/gemma4-soundness-and-punica-bench |
| Paper-track notebook | https://www.kaggle.com/code/moranzavdi/from-local-validation-to-live-evaluation |
| Competition | https://www.kaggle.com/competitions/gemma-4-developer-agent |
| Pages article | https://aeterna-data-ai-ml-research.github.io/Kaggle-Gemma-4-Developer-Agent-Competition/ |
| Medium | https://medium.com/@when1go2market |

Written by **Moran Zavdi**. If this notebook helped you, please star/cite the GitHub evidence release and credit the linked datasets/notebooks above.
